# Advanced RAG + RAGAS 평가 실습


<개선 전략>

데이터 의존성 최소에 중점을 두고 Modular구조 + Agentic RAG

쿼리 최적화 : Multi-Query Expansion / Query Rewrite / LLM Agent Router

증강 : 반복적 검색 (Iterative Retrieval)

생성 : LLM-based Re-ranking / Self-Critique


-모듈 구조-

query_analyzer가 질문 유형과 검색 필요 여부를 판단합니다.

query_rewriter가 검색용 질문을 재작성합니다.

multi_query_generator가 2~4개의 보조 질문을 만듭니다.

retriever_hub가 PDF/CSV/웹 retriever 중 적절한 것을 선택합니다.

reranker_or_filter가 검색 결과를 다시 정렬하거나 잡음을 제거합니다.

answerer가 최종 답변을 생성합니다.

critic가 근거 부족 여부를 판단하고 필요 시 재검색을 트리거합니다.


-에이전트 흐름-

질문 분석 -> 검색 필요 여부 결정 -> 적절한 retriever 선택 -> query rewrite -> multi-query retrieval -> 결과 병합 -> LLM re-rank -> 답변 생성 -> self-critique -> 필요 시 1회 재검색


-데이터 소스 라우팅-

Demian.pdf 관련 묘사/서술 질문은 PDF retriever
titanic.csv 관련 수치/행 기반 질문은 CSV retriever
최신 기사/웹 질문은 Web retriever
일반 상식형 질문은 검색 없이 LLM direct answer 또는 “이 노트북 범위 밖” 응답

-답변 정책-

항상 최종 답변 + 사용한 근거 요약 + 선택된 경로(router decision)를 함께 출력합니다.
Agent RAG 학습 목적상, 내부 판단 결과를 최소한으로 노출하는 것이 좋습니다.



<평가 전략>
- Naive RAG 와 Improved Modular+Agentic RAG 를 동시에 평가
- 정량평가 : RAGAS 4대 핵심 지표
- 정성평가 : LLM-as-a-judge
- 20개 이상 질문셋 기준으로 통계 요약 제공



### Step 0 : 최신 라이브러리 설치와 준비

- 충돌이 심하면 새 Colab 런타임에서 시작하는 것을 권장
- 설치 직후 import 에러가 나면 런타임 다시 시작 후 처음부터 실행


In [ ]:
# Colab 기본 패키지와 충돌하는 항목은 먼저 정리
!pip uninstall -y \
  langchain langchain-core langchain-community langchain-openai langchain-text-splitters \
  langchain-classic ragas chromadb faiss-cpu google-adk langgraph langgraph-prebuilt \
  > /dev/null 2>&1

In [ ]:
# 나머지 필요한 패키지만 설치
!pip install -q -U --upgrade-strategy only-if-needed \
  langchain \
  langchain-core \
  langchain-openai \
  langchain-community \
  langchain-text-splitters \
  langchain-classic \
  ragas \
  datasets \
  nest_asyncio \
  tiktoken \
  pypdf \
  beautifulsoup4 \
  lxml

In [ ]:
# RAGAS 는 비동기 루프를 사용하므로 notebook 환경에서는 nest_asyncio 적용
import nest_asyncio
nest_asyncio.apply()


In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# OpenAI API 키는 Colab userdata 에서만 읽기
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get("OpenAIapi")
os.environ['USER_AGENT'] = 'colab-advanced-rag-ragas/1.0'


In [ ]:
# 공통 경로 설정
DRIVE_BASE = "/content/drive/MyDrive"
PDF_PATH = f"{DRIVE_BASE}/Demian.pdf"
CSV_PATH = f"{DRIVE_BASE}/titanic.csv"
TEXT_PATH = f"{DRIVE_BASE}/state_of_the_union.txt"
WEB_URL = "https://it.chosun.com/news/articleView.html?idxno=2023092111831"

print("PDF exists :", os.path.exists(PDF_PATH))
print("CSV exists :", os.path.exists(CSV_PATH))
print("TXT exists :", os.path.exists(TEXT_PATH))


PDF exists : True
CSV exists : True
TXT exists : True


In [ ]:
# 주요 모듈 import
from langchain_community.document_loaders import PyPDFLoader, CSVLoader, WebBaseLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_classic.chains import RetrievalQA
from langchain_core.documents import Document

from IPython.display import Markdown, display
import tiktoken
import json
import re
from collections import OrderedDict


### Step 1 : 공통 구성요소 준비

- 토큰 길이 함수
- 공통 LLM / Embedding
- 데이터 로드
- Baseline 과 Improved 가 같은 데이터와 같은 모델 계열을 쓰게 맞춤


In [ ]:
# 토큰 길이 함수
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    return len(tokenizer.encode(text))


In [ ]:
# 공통 모델
common_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
common_embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# 평가용 judge / ragas 모델도 같은 계열로 통일
evaluator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
evaluator_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


In [ ]:
# 데이터 로드
pdf_loader = PyPDFLoader(PDF_PATH)
pdf_pages = pdf_loader.load()

csv_loader = CSVLoader(CSV_PATH)
csv_rows = csv_loader.load()

with open(TEXT_PATH, encoding="utf-8") as f:
    state_text = f.read()

try:
    web_loader = WebBaseLoader(WEB_URL)
    web_documents = web_loader.load()
except Exception as e:
    web_documents = []
    print("web load skipped :", e)

print("pdf pages :", len(pdf_pages))
print("csv rows :", len(csv_rows))
print("web docs :", len(web_documents))


pdf pages : 182
csv rows : 891
web docs : 1


### Step 2 : Baseline RAG 재구성

- 단일 저장소 기반 naive baseline
- PDF / CSV / Web 문서를 한 벡터스토어에 함께 넣음
- 검색 경로 분기 없이 그대로 RetrievalQA 사용


In [ ]:
# Baseline 문서 준비
baseline_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    length_function=tiktoken_len
)

baseline_pdf_docs = baseline_splitter.split_documents(pdf_pages)
baseline_csv_docs = baseline_splitter.split_documents(csv_rows)
baseline_web_docs = baseline_splitter.split_documents(web_documents) if web_documents else []

for doc in baseline_pdf_docs:
    doc.metadata["source_type"] = "pdf"
for doc in baseline_csv_docs:
    doc.metadata["source_type"] = "csv"
for doc in baseline_web_docs:
    doc.metadata["source_type"] = "web"

baseline_docs = baseline_pdf_docs + baseline_csv_docs + baseline_web_docs

baseline_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
baseline_embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
baseline_db = InMemoryVectorStore.from_documents(documents=baseline_docs, embedding=baseline_embedding_model)

baseline_qa = RetrievalQA.from_chain_type(
    baseline_llm,
    chain_type="stuff",
    retriever=baseline_db.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 4, "fetch_k": 12}
    ),
    return_source_documents=True
)


### Step 3 : Improved Agentic RAG 재구성

- 소스별 저장소를 분리
- 라우팅, 쿼리 재작성, 다중 질의, 리랭킹, 자기 점검을 포함
- 검색이 필요 없으면 direct answer 경로로 분기


In [ ]:
# Improved 문서 준비
improved_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=120,
    length_function=tiktoken_len
)

pdf_docs = improved_splitter.split_documents(pdf_pages)
csv_docs = improved_splitter.split_documents(csv_rows)
web_docs = improved_splitter.split_documents(web_documents) if web_documents else []

for doc in pdf_docs:
    doc.metadata["source_type"] = "pdf"
for doc in csv_docs:
    doc.metadata["source_type"] = "csv"
for doc in web_docs:
    doc.metadata["source_type"] = "web"

pdf_store = InMemoryVectorStore.from_documents(documents=pdf_docs, embedding=common_embedding_model)
csv_store = InMemoryVectorStore.from_documents(documents=csv_docs, embedding=common_embedding_model)
web_store = InMemoryVectorStore.from_documents(documents=web_docs, embedding=common_embedding_model) if web_docs else None


In [ ]:
# 보조 함수
def extract_json_block(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        return match.group(0)
    return text

def safe_json_loads(text, default):
    try:
        return json.loads(extract_json_block(text))
    except Exception:
        return default

def unique_docs(docs):
    seen = OrderedDict()
    for doc in docs:
        key = (doc.page_content[:300], tuple(sorted(doc.metadata.items())))
        if key not in seen:
            seen[key] = doc
    return list(seen.values())


In [ ]:
# Agentic 모듈
def query_analyzer(question):
    lowered = question.lower()

    if any(word in lowered for word in ["demian", "데미안", "character", "외모", "appearance", "sinclair"]):
        heuristic_route = "pdf"
    elif any(word in lowered for word in ["titanic", "승객", "fare", "age", "class", "survived", "csv", "객실", "나이", "성별"]):
        heuristic_route = "csv"
    elif any(word in lowered for word in ["news", "article", "website", "web", "chosun", "기사", "웹"]):
        heuristic_route = "web"
    elif any(word in lowered for word in ["안녕", "hello", "recipe", "레시피", "joke", "poem"]):
        heuristic_route = "no_retrieval"
    else:
        heuristic_route = "unknown"

    prompt = f"""
    You are a routing module for a modular RAG system.
    Decide whether retrieval is needed and which route is best.

    Available routes:
    - pdf
    - csv
    - web
    - no_retrieval

    Return JSON only.
    {{
      "needs_retrieval": true or false,
      "route": "pdf" | "csv" | "web" | "no_retrieval",
      "reason": "short explanation"
    }}

    Heuristic hint: {heuristic_route}
    Question: {question}
    """

    raw = common_llm.invoke(prompt).content
    parsed = safe_json_loads(raw, {
        "needs_retrieval": heuristic_route != "no_retrieval",
        "route": "pdf" if heuristic_route == "unknown" else heuristic_route,
        "reason": "fallback decision"
    })

    if parsed.get("route") == "web" and web_store is None:
        parsed["route"] = "no_retrieval"
        parsed["needs_retrieval"] = False
        parsed["reason"] = "web store unavailable, fallback to direct answer"

    return parsed

def query_rewriter(question, route):
    prompt = f"""
    Rewrite the question to improve retrieval quality.
    Keep the meaning the same.
    Route: {route}

    Return JSON only.
    {{
      "rewritten_query": "...",
      "rewrite_reason": "..."
    }}

    Question: {question}
    """
    raw = common_llm.invoke(prompt).content
    return safe_json_loads(raw, {
        "rewritten_query": question,
        "rewrite_reason": "fallback to original"
    })

def multi_query_generator(question, route, n=3):
    prompt = f"""
    Generate {n} alternative retrieval queries.
    Preserve intent but vary wording.
    Route: {route}

    Return JSON only.
    {{
      "queries": ["q1", "q2", "q3"]
    }}

    Question: {question}
    """
    raw = common_llm.invoke(prompt).content
    parsed = safe_json_loads(raw, {"queries": [question]})
    queries = [q.strip() for q in parsed.get("queries", [question]) if q.strip()]
    if question not in queries:
        queries.insert(0, question)
    return queries[: max(n + 1, 2)]

def retriever_router(route):
    if route == "pdf":
        return pdf_store.as_retriever(search_kwargs={"k": 5})
    if route == "csv":
        return csv_store.as_retriever(search_kwargs={"k": 5})
    if route == "web" and web_store is not None:
        return web_store.as_retriever(search_kwargs={"k": 5})
    return None

def retrieval_aggregator(queries, route):
    retriever = retriever_router(route)
    if retriever is None:
        return []
    gathered = []
    for q in queries:
        try:
            gathered.extend(retriever.invoke(q))
        except Exception:
            continue
    return unique_docs(gathered)

def llm_reranker(question, docs, top_n=4):
    if not docs:
        return []

    condensed = []
    short_docs = docs[:8]
    for idx, doc in enumerate(short_docs, start=1):
        snippet = doc.page_content[:700].replace("\n", " ")
        condensed.append(f"[{idx}] ({doc.metadata.get('source_type', 'unknown')}) {snippet}")

    prompt = f"""
    Rank the candidate documents for answering the question.
    Return JSON only.
    {{
      "ranked_ids": [1, 2, 3],
      "reason": "short explanation"
    }}

    Question: {question}

    Candidates:
    {chr(10).join(condensed)}
    """
    raw = common_llm.invoke(prompt).content
    parsed = safe_json_loads(raw, {"ranked_ids": list(range(1, len(short_docs)+1)), "reason": "fallback"})

    ranked = []
    for idx in parsed.get("ranked_ids", []):
        if 1 <= idx <= len(short_docs):
            ranked.append(short_docs[idx - 1])

    return unique_docs(ranked)[:top_n]

def answer_generator(question, route, docs):
    if not docs:
        return "참고할 검색 문서가 부족하여 근거 기반 답변을 만들지 못했습니다."

    context_text = "\n\n".join([
        f"[Context {idx}] source={doc.metadata.get('source_type', 'unknown')}\n{doc.page_content[:1200]}"
        for idx, doc in enumerate(docs, start=1)
    ])

    prompt = f"""
    당신은 근거 기반 RAG 답변 생성기입니다.
    반드시 제공된 context 안의 정보만 사용하세요.
    근거가 부족하면 부족하다고 분명히 말하세요.
    답변은 한국어로 작성하세요.

    Route: {route}
    Question: {question}

    Context:
    {context_text}
    """
    return common_llm.invoke(prompt).content

def critic(question, answer, docs):
    context_preview = "\n\n".join([doc.page_content[:500] for doc in docs[:3]]) if docs else ""
    prompt = f"""
    Review the current answer for groundedness.
    Decide whether one more retrieval attempt is needed.

    Return JSON only.
    {{
      "needs_retry": true or false,
      "critique": "short explanation",
      "better_query": "improved retrieval query"
    }}

    Question: {question}
    Current answer: {answer}
    Context preview: {context_preview}
    """
    raw = common_llm.invoke(prompt).content
    return safe_json_loads(raw, {
        "needs_retry": False,
        "critique": "fallback no retry",
        "better_query": question
    })

def direct_answer(question):
    prompt = f"""
    아래 질문에 한국어로 짧고 자연스럽게 답하세요.
    이 노트북의 데이터 범위를 벗어나면 그 점을 간단히 밝혀도 됩니다.

    Question: {question}
    """
    return common_llm.invoke(prompt).content

def run_modular_agentic_rag(question):
    analysis = query_analyzer(question)
    route = analysis.get("route", "no_retrieval")

    if not analysis.get("needs_retrieval", True) or route == "no_retrieval":
        return {
            "question": question,
            "route": route,
            "analysis": analysis,
            "rewritten_query": question,
            "queries_used": [question],
            "retried": False,
            "critique": "retrieval skipped",
            "documents": [],
            "answer": direct_answer(question)
        }

    rewrite_info = query_rewriter(question, route)
    rewritten_query = rewrite_info.get("rewritten_query", question)
    queries = multi_query_generator(rewritten_query, route, n=3)

    retrieved_docs = retrieval_aggregator(queries, route)
    ranked_docs = llm_reranker(question, retrieved_docs, top_n=4)
    answer = answer_generator(question, route, ranked_docs)
    critique_info = critic(question, answer, ranked_docs)

    retried = False
    if critique_info.get("needs_retry", False):
        retried = True
        better_query = critique_info.get("better_query", rewritten_query)
        retry_queries = multi_query_generator(better_query, route, n=2)
        retrieved_docs = retrieval_aggregator(retry_queries, route)
        ranked_docs = llm_reranker(question, retrieved_docs, top_n=4)
        answer = answer_generator(question, route, ranked_docs)

    return {
        "question": question,
        "route": route,
        "analysis": analysis,
        "rewritten_query": rewritten_query,
        "queries_used": queries,
        "retried": retried,
        "critique": critique_info.get("critique", ""),
        "documents": ranked_docs,
        "answer": answer
    }

def run_baseline(question):
    result = baseline_qa.invoke({"query": question})
    return {
        "answer": result["result"],
        "documents": result.get("source_documents", [])
    }


### Step 4 : 평가 질문셋 구성

- 평가는 소수 예시가 아니라 20개 이상 수동 질문셋 사용
- 질문 카테고리를 나누어 category별 평균도 계산
- reference 는 사람이 직접 적은 모범답안


In [ ]:
# 20개 수동 평가셋
import pandas as pd

evaluation_records = [
    {
        "category": "pdf",
        "question": "데미안의 외모는 본문에서 어떻게 묘사되나요?",
        "reference": "데미안은 인상적인 눈빛과 뚜렷한 분위기를 지닌 인물로 묘사되며, 일반적인 소년과는 다른 신비롭고 강렬한 인상을 줍니다. 구체적 표현은 문맥에 따라 다르므로 본문 근거를 바탕으로 답해야 합니다."
    },
    {
        "category": "pdf",
        "question": "데미안과 싱클레어의 관계를 설명해줘.",
        "reference": "데미안은 싱클레어에게 큰 영향을 주는 인물이며, 싱클레어의 내면 성장과 자아 탐색을 이끄는 역할을 합니다."
    },
    {
        "category": "pdf",
        "question": "싱클레어의 성장 서사에서 데미안은 어떤 역할을 하나요?",
        "reference": "데미안은 싱클레어가 기존 가치관을 의심하고 더 넓은 자아 인식으로 나아가게 돕는 촉진자 역할을 합니다."
    },
    {
        "category": "pdf",
        "question": "데미안이라는 작품의 분위기는 어떤 식으로 전달되나요?",
        "reference": "작품은 내면 탐구, 상징, 불안과 각성의 분위기를 통해 철학적이고 성찰적인 정서를 전달합니다."
    },
    {
        "category": "pdf",
        "question": "본문에서 데미안은 평범한 인물처럼 보이나요, 특별한 인물처럼 보이나요?",
        "reference": "데미안은 평범하기보다 특별하고 이질적인 인상으로 제시되며, 싱클레어에게 강한 영향을 주는 존재로 보입니다."
    },
    {
        "category": "csv",
        "question": "타이타닉 CSV에서 승객의 성별을 확인할 수 있는 필드는 무엇인가요?",
        "reference": "승객의 성별은 보통 sex 필드로 확인합니다."
    },
    {
        "category": "csv",
        "question": "타이타닉 CSV에서 나이를 확인하는 데 쓰는 컬럼은 무엇인가요?",
        "reference": "승객의 나이는 age 컬럼으로 확인합니다."
    },
    {
        "category": "csv",
        "question": "타이타닉 CSV에서 객실 등급을 나타내는 항목은 무엇인가요?",
        "reference": "객실 등급은 pclass 컬럼으로 표현됩니다."
    },
    {
        "category": "csv",
        "question": "생존 여부를 나타내는 필드는 무엇인가요?",
        "reference": "생존 여부는 survived 필드로 확인합니다."
    },
    {
        "category": "csv",
        "question": "요금 정보를 보는 컬럼은 무엇인가요?",
        "reference": "승객 요금은 fare 컬럼으로 확인합니다."
    },
    {
        "category": "csv",
        "question": "타이타닉 CSV에서 승객의 인구통계 정보를 보여주는 대표 컬럼들을 말해줘.",
        "reference": "대표적으로 pclass, sex, age 같은 컬럼이 승객의 인구통계적 특성을 보여줍니다."
    },
    {
        "category": "routing",
        "question": "데미안의 외모를 묻는 질문은 어떤 검색 경로를 선택해야 하나요?",
        "reference": "데미안의 외모를 묻는 질문은 소설 본문이 있는 PDF 경로를 선택하는 것이 적절합니다."
    },
    {
        "category": "routing",
        "question": "타이타닉 CSV 컬럼 의미를 묻는 질문은 어느 소스로 가야 하나요?",
        "reference": "타이타닉 CSV 구조를 묻는 질문은 CSV 소스로 라우팅하는 것이 맞습니다."
    },
    {
        "category": "routing",
        "question": "웹 기사 관련 질문은 어떤 검색 경로를 타야 하나요?",
        "reference": "웹 기사 관련 질문은 웹 문서를 대상으로 하는 경로를 선택해야 합니다."
    },
    {
        "category": "routing",
        "question": "검색이 필요 없는 짧은 인사 요청은 어떻게 처리해야 하나요?",
        "reference": "짧은 인사 요청은 retrieval 없이 direct answer 경로로 처리하는 것이 적절합니다."
    },
    {
        "category": "routing",
        "question": "카레 레시피를 물어보면 이 노트북의 검색 데이터로 답해야 하나요?",
        "reference": "카레 레시피는 이 노트북의 PDF·CSV·웹 기사 범위 밖이므로 retrieval 없이 제한적으로 답하거나 범위 밖임을 밝히는 것이 적절합니다."
    },
    {
        "category": "no_retrieval",
        "question": "안녕이라고 짧게 인사해줘.",
        "reference": "짧은 인사는 파일 검색 없이 간단히 응답하면 됩니다."
    },
    {
        "category": "no_retrieval",
        "question": "이 노트북 데이터와 무관한 일반 상식 질문은 어떻게 답하는 게 좋나요?",
        "reference": "데이터 범위 밖 질문은 검색 없이 답하되, 노트북 데이터 범위를 벗어난다는 점을 함께 안내하는 것이 좋습니다."
    },
    {
        "category": "no_retrieval",
        "question": "검색 데이터를 쓰지 말고 짧게 자기소개해줘.",
        "reference": "검색 없이도 가능한 짧은 자기소개성 응답을 제공하면 됩니다."
    },
    {
        "category": "no_retrieval",
        "question": "카레 레시피를 간단히 알려줘.",
        "reference": "카레 레시피는 이 노트북의 데이터 범위 밖이므로, 검색 기반 답변 대신 일반적인 짧은 안내나 범위 제한 설명을 해야 합니다."
    },
    {
        "category": "pdf",
        "question": "데미안이 싱클레어에게 주는 영향은 긍정적인가요?",
        "reference": "데미안은 싱클레어에게 혼란과 도전을 주기도 하지만 궁극적으로 자아 성장을 촉진하는 긍정적 영향으로 해석할 수 있습니다."
    },
    {
        "category": "csv",
        "question": "타이타닉 CSV에서 승객 식별에 도움이 되는 컬럼 예시를 말해줘.",
        "reference": "이름, 객실 등급, 성별, 나이, 요금 같은 컬럼이 승객 식별과 특성 파악에 도움이 됩니다."
    },
    {
        "category": "routing",
        "question": "소설 인물 묘사 질문과 CSV 컬럼 질문은 같은 경로로 처리하면 안 되는 이유는 무엇인가요?",
        "reference": "두 질문은 데이터 형식과 필요한 근거가 다르므로, 소설 질문은 PDF 경로, CSV 질문은 표 구조 경로로 분리해야 검색 품질이 좋아집니다."
    },
    {
        "category": "no_retrieval",
        "question": "검색 자료를 사용하지 말고 간단한 농담 하나 해줘.",
        "reference": "이런 요청은 retrieval이 필요 없고 direct answer 경로가 적절합니다."
    }
]

eval_df = pd.DataFrame(evaluation_records)
print("evaluation set size :", len(eval_df))
display(eval_df.head())

evaluation set size : 24


,category,question,reference
0,pdf,데미안의 외모는 본문에서 어떻게 묘사되나요?,"데미안은 인상적인 눈빛과 뚜렷한 분위기를 지닌 인물로 묘사되며, 일반적인 소년과는 ..."
1,pdf,데미안과 싱클레어의 관계를 설명해줘.,"데미안은 싱클레어에게 큰 영향을 주는 인물이며, 싱클레어의 내면 성장과 자아 탐색을..."
2,pdf,싱클레어의 성장 서사에서 데미안은 어떤 역할을 하나요?,데미안은 싱클레어가 기존 가치관을 의심하고 더 넓은 자아 인식으로 나아가게 돕는 촉...
3,pdf,데미안이라는 작품의 분위기는 어떤 식으로 전달되나요?,"작품은 내면 탐구, 상징, 불안과 각성의 분위기를 통해 철학적이고 성찰적인 정서를 ..."
4,pdf,"본문에서 데미안은 평범한 인물처럼 보이나요, 특별한 인물처럼 보이나요?","데미안은 평범하기보다 특별하고 이질적인 인상으로 제시되며, 싱클레어에게 강한 영향을..."


### Step 5 : Baseline / Improved 응답과 문맥 수집

- 각 질문에 대해 두 시스템을 모두 실행
- 응답과 retrieved_contexts 를 dataset 형태로 정리
- 이후 baseline / improved 를 RAGAS 로 각각 평가


In [ ]:
# retrieved_contexts 를 문자열 리스트로 변환
def docs_to_contexts(docs):
    return [doc.page_content for doc in docs] if docs else []

def source_types(docs):
    return [doc.metadata.get("source_type", "unknown") for doc in docs]

baseline_eval_rows = []
improved_eval_rows = []
combined_rows = []

for row in evaluation_records:
    question = row["question"]
    category = row["category"]
    reference = row["reference"]

    baseline_out = run_baseline(question)
    improved_out = run_modular_agentic_rag(question)

    baseline_contexts = docs_to_contexts(baseline_out["documents"])
    improved_contexts = docs_to_contexts(improved_out["documents"])

    baseline_eval_rows.append({
        "category": category,
        "user_input": question,
        "response": baseline_out["answer"],
        "retrieved_contexts": baseline_contexts,
        "reference": reference
    })

    improved_eval_rows.append({
        "category": category,
        "user_input": question,
        "response": improved_out["answer"],
        "retrieved_contexts": improved_contexts,
        "reference": reference
    })

    combined_rows.append({
        "category": category,
        "question": question,
        "reference": reference,
        "baseline_answer": baseline_out["answer"],
        "improved_answer": improved_out["answer"],
        "baseline_source_types": source_types(baseline_out["documents"]),
        "improved_source_types": source_types(improved_out["documents"]),
        "improved_route": improved_out["route"],
        "improved_retried": improved_out["retried"],
        "improved_queries": improved_out["queries_used"],
        "improved_critique": improved_out["critique"]
    })

baseline_eval_df = pd.DataFrame(baseline_eval_rows)
improved_eval_df = pd.DataFrame(improved_eval_rows)
comparison_df = pd.DataFrame(combined_rows)

print("baseline rows :", len(baseline_eval_df))
print("improved rows :", len(improved_eval_df))
display(comparison_df.head())


baseline rows : 24
improved rows : 24


,category,question,reference,baseline_answer,improved_answer,baseline_source_types,improved_source_types,improved_route,improved_retried,improved_queries,improved_critique
0,pdf,데미안의 외모는 본문에서 어떻게 묘사되나요?,"데미안은 인상적인 눈빛과 뚜렷한 분위기를 지닌 인물로 묘사되며, 일반적인 소년과는 ...","본문에서는 데미안의 외모에 대한 구체적인 묘사가 없습니다. 대신, 화자는 데미안의 ...",제공된 본문에서는 데미안의 외모에 대한 구체적인 묘사가 없습니다. 데미안의 외모에 ...,"[web, pdf, csv, pdf]","[pdf, pdf, pdf]",pdf,True,"[본문에서 데미안의 외모는 어떻게 설명되고 있나요?, 데미안의 외모에 대한 설명은 ...",The current answer provides a detailed descrip...
1,pdf,데미안과 싱클레어의 관계를 설명해줘.,"데미안은 싱클레어에게 큰 영향을 주는 인물이며, 싱클레어의 내면 성장과 자아 탐색을...",데미안과 싱클레어의 관계는 깊은 정신적 유대와 상호 영향을 기반으로 합니다. 싱클레...,데미안과 싱클레어의 관계는 매우 복잡하고 깊은 정신적 유대감으로 설명될 수 있습니다...,"[web, pdf, csv, web]","[pdf, pdf, pdf]",pdf,False,"[데미안과 싱클레어의 관계에 대해 설명해 주세요., 데미안과 싱클레어의 관계를 설명...",The current answer provides a comprehensive ex...
2,pdf,싱클레어의 성장 서사에서 데미안은 어떤 역할을 하나요?,데미안은 싱클레어가 기존 가치관을 의심하고 더 넓은 자아 인식으로 나아가게 돕는 촉...,데미안은 싱클레어의 성장 서사에서 중요한 역할을 합니다. 그는 싱클레어에게 자신의 ...,데미안은 싱클레어의 성장 서사에서 중요한 역할을 합니다. 그는 싱클레어에게 내면의 ...,"[pdf, web, csv, web]","[pdf, pdf, pdf]",pdf,True,"[데미안은 싱클레어의 성장 서사에서 어떤 역할을 수행하나요?, 데미안이 싱클레어의 ...",The current answer provides a general overview...
3,pdf,데미안이라는 작품의 분위기는 어떤 식으로 전달되나요?,"작품은 내면 탐구, 상징, 불안과 각성의 분위기를 통해 철학적이고 성찰적인 정서를 ...","""데미안""이라는 작품의 분위기는 주로 주인공의 내면적 갈등과 성장 과정을 통해 전달...","""데미안""이라는 작품의 분위기는 주로 인물들의 내면적 갈등과 신비로운 요소를 통해 ...","[pdf, pdf, pdf, pdf]","[pdf, pdf, pdf]",pdf,True,"[데미안 작품의 분위기는 어떻게 표현되고 전달되나요?, 데미안의 분위기는 어떤 방식...",The current answer provides a general overview...
4,pdf,"본문에서 데미안은 평범한 인물처럼 보이나요, 특별한 인물처럼 보이나요?","데미안은 평범하기보다 특별하고 이질적인 인상으로 제시되며, 싱클레어에게 강한 영향을...","본문에서 데미안은 평범한 인물처럼 보이지 않고, 특별한 인물처럼 묘사됩니다. 그는 ...","본문에서 데미안은 평범한 인물처럼 보이지 않고, 오히려 특별한 인물로 묘사됩니다. ...","[pdf, pdf, pdf, pdf]","[pdf, pdf, pdf]",pdf,True,"[본문에서 데미안은 평범한 인물인지, 아니면 특별한 인물인지에 대한 설명은 무엇인가...",The current answer provides a general interpre...


### Step 6 : RAGAS 정량평가

- baseline(naive) 과 improved 를 각각 RAGAS 로 평가
- 평균만 보지 않고 질문별 점수, 표준편차, category별 평균도 함께 확인
- 소규모 예시가 아니라 20개 이상 질문셋을 기준으로 비교


In [ ]:
from datasets import Dataset
# Dataset 변환
baseline_dataset = Dataset.from_pandas(baseline_eval_df[["user_input", "response", "retrieved_contexts", "reference"]], preserve_index=False)
improved_dataset = Dataset.from_pandas(improved_eval_df[["user_input", "response", "retrieved_contexts", "reference"]], preserve_index=False)

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

# Baseline RAGAS 평가
baseline_result = evaluate(
    dataset=baseline_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    raise_exceptions=False
)

baseline_scores_df = baseline_result.to_pandas()
baseline_scores_df = pd.concat([baseline_eval_df[["category", "user_input"]].reset_index(drop=True), baseline_scores_df.reset_index(drop=True)], axis=1)
display(baseline_scores_df.head())

/tmp/ipykernel_22631/2629866954.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/tmp/ipykernel_22631/2629866954.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/tmp/ipykernel_22631/2629866954.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import faithfu

Evaluating:   0%|          | 0/96 [00:00<?, ?it/s]

,category,user_input,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,pdf,데미안의 외모는 본문에서 어떻게 묘사되나요?,데미안의 외모는 본문에서 어떻게 묘사되나요?,"[모두의연구소 ‘AI학교 아이펠’, ICLR 2024 워크숍 혁신 기술 논문 채택 ...","본문에서는 데미안의 외모에 대한 구체적인 묘사가 없습니다. 대신, 화자는 데미안의 ...","데미안은 인상적인 눈빛과 뚜렷한 분위기를 지닌 인물로 묘사되며, 일반적인 소년과는 ...",1.0,0.000000,0.5,0.5
1,pdf,데미안과 싱클레어의 관계를 설명해줘.,데미안과 싱클레어의 관계를 설명해줘.,[파이낸스\n\n금융\n증권\n핀테크·블록체인\n\n\n칼럼·인터뷰\n\n칼럼\n기...,데미안과 싱클레어의 관계는 깊은 정신적 유대와 상호 영향을 기반으로 합니다. 싱클레...,"데미안은 싱클레어에게 큰 영향을 주는 인물이며, 싱클레어의 내면 성장과 자아 탐색을...",1.0,0.259822,0.5,1.0
2,pdf,싱클레어의 성장 서사에서 데미안은 어떤 역할을 하나요?,싱클레어의 성장 서사에서 데미안은 어떤 역할을 하나요?,[DEMIAN \nwe gained a critical understanding o...,데미안은 싱클레어의 성장 서사에서 중요한 역할을 합니다. 그는 싱클레어에게 자신의 ...,데미안은 싱클레어가 기존 가치관을 의심하고 더 넓은 자아 인식으로 나아가게 돕는 촉...,1.0,0.237532,1.0,1.0
3,pdf,데미안이라는 작품의 분위기는 어떤 식으로 전달되나요?,데미안이라는 작품의 분위기는 어떤 식으로 전달되나요?,[DJ:MIAN \none day in the early morning class ...,"""데미안""이라는 작품의 분위기는 주로 주인공의 내면적 갈등과 성장 과정을 통해 전달...","작품은 내면 탐구, 상징, 불안과 각성의 분위기를 통해 철학적이고 성찰적인 정서를 ...",0.5,0.195268,1.0,1.0
4,pdf,"본문에서 데미안은 평범한 인물처럼 보이나요, 특별한 인물처럼 보이나요?","본문에서 데미안은 평범한 인물처럼 보이나요, 특별한 인물처럼 보이나요?","[DEMIAN \nwith a feeling of nausea, I noticed ...","본문에서 데미안은 평범한 인물처럼 보이지 않고, 특별한 인물처럼 묘사됩니다. 그는 ...","데미안은 평범하기보다 특별하고 이질적인 인상으로 제시되며, 싱클레어에게 강한 영향을...",1.0,0.330573,1.0,1.0


In [ ]:
# Improved RAGAS 평가
improved_result = evaluate(
    dataset=improved_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    raise_exceptions=False
)

improved_scores_df = improved_result.to_pandas()
improved_scores_df = pd.concat([improved_eval_df[["category", "user_input"]].reset_index(drop=True), improved_scores_df.reset_index(drop=True)], axis=1)
display(improved_scores_df.head())


Evaluating:   0%|          | 0/96 [00:00<?, ?it/s]

,category,user_input,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,pdf,데미안의 외모는 본문에서 어떻게 묘사되나요?,데미안의 외모는 본문에서 어떻게 묘사되나요?,[DJ:MIAN \none day in the early morning class ...,제공된 본문에서는 데미안의 외모에 대한 구체적인 묘사가 없습니다. 데미안의 외모에 ...,"데미안은 인상적인 눈빛과 뚜렷한 분위기를 지닌 인물로 묘사되며, 일반적인 소년과는 ...",1.0,0.000000,0.833333,0.5
1,pdf,데미안과 싱클레어의 관계를 설명해줘.,데미안과 싱클레어의 관계를 설명해줘.,[DJ:MIAN \none day in the early morning class ...,데미안과 싱클레어의 관계는 매우 복잡하고 깊은 정신적 유대감으로 설명될 수 있습니다...,"데미안은 싱클레어에게 큰 영향을 주는 인물이며, 싱클레어의 내면 성장과 자아 탐색을...",1.0,0.239723,1.000000,1.0
2,pdf,싱클레어의 성장 서사에서 데미안은 어떤 역할을 하나요?,싱클레어의 성장 서사에서 데미안은 어떤 역할을 하나요?,[DEMIAN \nthe organ or in front of the fire. W...,데미안은 싱클레어의 성장 서사에서 중요한 역할을 합니다. 그는 싱클레어에게 내면의 ...,데미안은 싱클레어가 기존 가치관을 의심하고 더 넓은 자아 인식으로 나아가게 돕는 촉...,1.0,0.237613,1.000000,1.0
3,pdf,데미안이라는 작품의 분위기는 어떤 식으로 전달되나요?,데미안이라는 작품의 분위기는 어떤 식으로 전달되나요?,"[EVA \nbuffeted by the wind, Demian himself op...","""데미안""이라는 작품의 분위기는 주로 인물들의 내면적 갈등과 신비로운 요소를 통해 ...","작품은 내면 탐구, 상징, 불안과 각성의 분위기를 통해 철학적이고 성찰적인 정서를 ...",1.0,0.235950,1.000000,1.0
4,pdf,"본문에서 데미안은 평범한 인물처럼 보이나요, 특별한 인물처럼 보이나요?","본문에서 데미안은 평범한 인물처럼 보이나요, 특별한 인물처럼 보이나요?","[DEMIAN \nwith a feeling of nausea, I noticed ...","본문에서 데미안은 평범한 인물처럼 보이지 않고, 오히려 특별한 인물로 묘사됩니다. ...","데미안은 평범하기보다 특별하고 이질적인 인상으로 제시되며, 싱클레어에게 강한 영향을...",1.0,0.330458,1.000000,1.0


In [ ]:
# metric 평균 비교
metric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

baseline_mean = baseline_scores_df[metric_cols].mean().to_frame(name="baseline_mean")
improved_mean = improved_scores_df[metric_cols].mean().to_frame(name="improved_mean")
summary_mean_df = baseline_mean.join(improved_mean)
summary_mean_df["delta"] = summary_mean_df["improved_mean"] - summary_mean_df["baseline_mean"]
display(summary_mean_df)


,baseline_mean,improved_mean,delta
faithfulness,0.545139,0.613542,0.068403
answer_relevancy,0.277790,0.244988,-0.032802
context_precision,0.429398,0.527778,0.098380
context_recall,0.770833,0.604167,-0.166667


In [ ]:
# metric 표준편차 비교
baseline_std = baseline_scores_df[metric_cols].std().to_frame(name="baseline_std")
improved_std = improved_scores_df[metric_cols].std().to_frame(name="improved_std")
summary_std_df = baseline_std.join(improved_std)
display(summary_std_df)


,baseline_std,improved_std
faithfulness,0.479646,0.459644
answer_relevancy,0.322254,0.318814
context_precision,0.415657,0.497983
context_recall,0.416485,0.488546


In [ ]:
# category별 평균 비교
baseline_category_df = baseline_scores_df.groupby("category")[metric_cols].mean().add_prefix("baseline_")
improved_category_df = improved_scores_df.groupby("category")[metric_cols].mean().add_prefix("improved_")
category_compare_df = baseline_category_df.join(improved_category_df)
display(category_compare_df)


,baseline_faithfulness,baseline_answer_relevancy,baseline_context_precision,baseline_context_recall,improved_faithfulness,improved_answer_relevancy,improved_context_precision,improved_context_recall
category,,,,,,,,
csv,0.857143,0.591159,0.686508,1.000000,0.934524,0.525610,0.714286,0.857143
no_retrieval,0.000000,0.161923,0.000000,0.400000,0.220000,0.025201,0.000000,0.400000
pdf,0.750000,0.211888,0.666667,0.916667,1.000000,0.211141,0.972222,0.916667
routing,0.430556,0.074650,0.250000,0.666667,0.180556,0.134596,0.305556,0.166667


In [ ]:
# 질문별 원시 점수 비교
question_level_compare_df = pd.DataFrame({
    "category": baseline_scores_df["category"].values.tolist(),
    "question": baseline_scores_df["user_input"].values.tolist(),
    "baseline_faithfulness": baseline_scores_df["faithfulness"].values.tolist(),
    "improved_faithfulness": improved_scores_df["faithfulness"].values.tolist(),
    "baseline_answer_relevancy": baseline_scores_df["answer_relevancy"].values.tolist(),
    "improved_answer_relevancy": improved_scores_df["answer_relevancy"].values.tolist(),
    "baseline_context_precision": baseline_scores_df["context_precision"].values.tolist(),
    "improved_context_precision": improved_scores_df["context_precision"].values.tolist(),
    "baseline_context_recall": baseline_scores_df["context_recall"].values.tolist(),
    "improved_context_recall": improved_scores_df["context_recall"].values.tolist()
})
display(question_level_compare_df)

,category,question,baseline_faithfulness,improved_faithfulness,baseline_answer_relevancy,improved_answer_relevancy,baseline_context_precision,improved_context_precision,baseline_context_recall,improved_context_recall
0,pdf,"[데미안의 외모는 본문에서 어떻게 묘사되나요?, 데미안의 외모는 본문에서 어떻게 묘...",1.000000,1.000000,0.000000,0.000000,0.500000,0.833333,0.5,0.5
1,pdf,"[데미안과 싱클레어의 관계를 설명해줘., 데미안과 싱클레어의 관계를 설명해줘.]",1.000000,1.000000,0.259822,0.239723,0.500000,1.000000,1.0,1.0
2,pdf,"[싱클레어의 성장 서사에서 데미안은 어떤 역할을 하나요?, 싱클레어의 성장 서사에서...",1.000000,1.000000,0.237532,0.237613,1.000000,1.000000,1.0,1.0
3,pdf,"[데미안이라는 작품의 분위기는 어떤 식으로 전달되나요?, 데미안이라는 작품의 분위기...",0.500000,1.000000,0.195268,0.235950,1.000000,1.000000,1.0,1.0
4,pdf,"[본문에서 데미안은 평범한 인물처럼 보이나요, 특별한 인물처럼 보이나요?, 본문에서...",1.000000,1.000000,0.330573,0.330458,1.000000,1.000000,1.0,1.0
5,csv,"[타이타닉 CSV에서 승객의 성별을 확인할 수 있는 필드는 무엇인가요?, 타이타닉 ...",1.000000,1.000000,0.999999,0.999999,1.000000,1.000000,1.0,1.0
6,csv,"[타이타닉 CSV에서 나이를 확인하는 데 쓰는 컬럼은 무엇인가요?, 타이타닉 CSV...",1.000000,1.000000,0.936254,0.936177,0.805556,1.000000,1.0,1.0
7,csv,"[타이타닉 CSV에서 객실 등급을 나타내는 항목은 무엇인가요?, 타이타닉 CSV에서...",1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.0
8,csv,"[생존 여부를 나타내는 필드는 무엇인가요?, 생존 여부를 나타내는 필드는 무엇인가요?]",1.000000,0.666667,0.434367,0.000000,0.500000,0.000000,1.0,0.0
9,csv,"[요금 정보를 보는 컬럼은 무엇인가요?, 요금 정보를 보는 컬럼은 무엇인가요?]",1.000000,1.000000,0.000000,0.000000,0.500000,0.000000,1.0,1.0


### Step 7 : LLM-as-a-judge 정성평가

- baseline 과 improved 답변을 pairwise 비교
- 단순 승자 판단만 하지 않고 이유와 약점도 함께 기록
- 한국어 총평 기준으로 qualitative evidence 확보


In [ ]:
# 정성평가 judge
def judge_pairwise(question, reference, baseline_answer, improved_answer):
    prompt = f"""
    당신은 RAG 시스템 평가자입니다.
    아래 질문에 대한 두 답변을 비교 평가하세요.

    평가 기준:
    1. 질문 적합성
    2. 근거 기반성
    3. 설명 명확성
    4. 실제 서비스 관점의 유용성

    reference answer:
    {reference}

    question:
    {question}

    baseline answer:
    {baseline_answer}

    improved answer:
    {improved_answer}

    JSON 으로만 답하세요.
    {{
      "winner": "baseline" | "improved" | "tie",
      "reason": "짧은 이유",
      "baseline_weakness": "baseline 약점",
      "improved_weakness": "improved 약점",
      "service_recommendation": "실서비스 관점 권고"
    }}
    """

    raw = evaluator_llm.invoke(prompt).content
    return safe_json_loads(raw, {
        "winner": "tie",
        "reason": "judge parsing fallback",
        "baseline_weakness": "N/A",
        "improved_weakness": "N/A",
        "service_recommendation": "추가 검토 필요"
    })


In [ ]:
# 전 질문 정성평가 수행
judge_rows = []
for row in combined_rows:
    judged = judge_pairwise(
        question=row["question"],
        reference=row["reference"],
        baseline_answer=row["baseline_answer"],
        improved_answer=row["improved_answer"]
    )
    judge_rows.append({
        "category": row["category"],
        "question": row["question"],
        "winner": judged["winner"],
        "reason": judged["reason"],
        "baseline_weakness": judged["baseline_weakness"],
        "improved_weakness": judged["improved_weakness"],
        "service_recommendation": judged["service_recommendation"]
    })

judge_df = pd.DataFrame(judge_rows)
display(judge_df.head())


,category,question,winner,reason,baseline_weakness,improved_weakness,service_recommendation
0,pdf,데미안의 외모는 본문에서 어떻게 묘사되나요?,baseline,baseline이 데미안의 외모에 대한 간접적인 묘사를 포함하고 있어 질문에 더 적...,구체적인 외모 묘사가 부족하여 독자가 원하는 정보를 완전히 제공하지 못함.,"답변이 너무 부정적이며, 질문에 대한 유용한 정보가 전혀 제공되지 않음.","질문에 대한 답변은 가능한 한 구체적인 정보를 제공하고, 간접적인 묘사도 포함하여 ..."
1,pdf,데미안과 싱클레어의 관계를 설명해줘.,improved,보다 구체적이고 깊이 있는 설명을 제공하여 관계의 복잡성을 잘 드러냄.,상대적으로 일반적인 설명으로 구체적인 예시가 부족함.,조금 더 간결하게 요약할 수 있었음.,정신적 유대와 상호 영향을 강조하는 콘텐츠를 제공하여 독자의 이해를 돕는 것이 좋음.
2,pdf,싱클레어의 성장 서사에서 데미안은 어떤 역할을 하나요?,improved,improved 답변이 더 깊이 있는 분석과 명확한 설명을 제공하여 질문에 대한 적...,"기본 답변은 다소 일반적이며, 구체적인 예시나 세부 사항이 부족함.","개선된 답변은 다소 길어질 수 있으며, 간결함이 떨어질 수 있음.",독자에게 더 많은 구체적 사례나 인용을 제공하여 이해를 돕는 것이 좋음.
3,pdf,데미안이라는 작품의 분위기는 어떤 식으로 전달되나요?,improved,보다 구체적이고 심도 있는 설명을 제공하여 작품의 분위기를 잘 전달함.,일반적인 설명에 그쳐 구체적인 예시가 부족함.,상징적 요소에 대한 설명이 다소 복잡하여 이해하기 어려울 수 있음.,독자에게 작품의 분위기를 효과적으로 전달하기 위해 구체적인 예시와 함께 명확한 설명...
4,pdf,"본문에서 데미안은 평범한 인물처럼 보이나요, 특별한 인물처럼 보이나요?",improved,improved 답변이 더 구체적이고 다양한 측면에서 데미안의 특별함을 설명하고 있...,"기본 답변은 데미안의 특별함을 언급하지만, 구체적인 예시나 설명이 부족하다.","개성이나 행동에 대한 설명이 다소 주관적일 수 있으며, 더 많은 근거가 필요할 수 있다.",독자에게 더 많은 맥락과 예시를 제공하여 인물의 특성을 명확히 이해할 수 있도록 하...


In [ ]:
# 정성평가 승자 분포
judge_summary = judge_df["winner"].value_counts(dropna=False).to_frame(name="count")
display(judge_summary)


,count
winner,
improved,15
baseline,6
tie,3


In [ ]:
# qualitative highlight 3개
for idx, row in judge_df.head(3).iterrows():
    display(Markdown(f"""
## Qualitative Highlight {idx + 1}

**질문**
{row['question']}

- category: {row['category']}
- winner: {row['winner']}
- reason: {row['reason']}
- baseline weakness: {row['baseline_weakness']}
- improved weakness: {row['improved_weakness']}
- service recommendation: {row['service_recommendation']}
"""))



## Qualitative Highlight 1

**질문**
데미안의 외모는 본문에서 어떻게 묘사되나요?

- category: pdf
- winner: baseline
- reason: baseline이 데미안의 외모에 대한 간접적인 묘사를 포함하고 있어 질문에 더 적합하다.
- baseline weakness: 구체적인 외모 묘사가 부족하여 독자가 원하는 정보를 완전히 제공하지 못함.
- improved weakness: 답변이 너무 부정적이며, 질문에 대한 유용한 정보가 전혀 제공되지 않음.
- service recommendation: 질문에 대한 답변은 가능한 한 구체적인 정보를 제공하고, 간접적인 묘사도 포함하여 독자의 이해를 돕는 것이 중요하다.



## Qualitative Highlight 2

**질문**
데미안과 싱클레어의 관계를 설명해줘.

- category: pdf
- winner: improved
- reason: 보다 구체적이고 깊이 있는 설명을 제공하여 관계의 복잡성을 잘 드러냄.
- baseline weakness: 상대적으로 일반적인 설명으로 구체적인 예시가 부족함.
- improved weakness: 조금 더 간결하게 요약할 수 있었음.
- service recommendation: 정신적 유대와 상호 영향을 강조하는 콘텐츠를 제공하여 독자의 이해를 돕는 것이 좋음.



## Qualitative Highlight 3

**질문**
싱클레어의 성장 서사에서 데미안은 어떤 역할을 하나요?

- category: pdf
- winner: improved
- reason: improved 답변이 더 깊이 있는 분석과 명확한 설명을 제공하여 질문에 대한 적합성이 높음.
- baseline weakness: 기본 답변은 다소 일반적이며, 구체적인 예시나 세부 사항이 부족함.
- improved weakness: 개선된 답변은 다소 길어질 수 있으며, 간결함이 떨어질 수 있음.
- service recommendation: 독자에게 더 많은 구체적 사례나 인용을 제공하여 이해를 돕는 것이 좋음.


### Step 8 : 최종 해석

- 평균 점수만 보면 안 됨
- category별 편차와 표준편차도 같이 봐야 함
- 정성평가를 통해 왜 improved 가 더 낫거나, 혹은 왜 차이가 작았는지를 해석


In [ ]:
# 최종 요약 표
final_report_df = summary_mean_df.copy()
final_report_df = final_report_df.join(summary_std_df)
display(final_report_df)


,baseline_mean,improved_mean,delta,baseline_std,improved_std
faithfulness,0.545139,0.613542,0.068403,0.479646,0.459644
answer_relevancy,0.277790,0.244988,-0.032802,0.322254,0.318814
context_precision,0.429398,0.527778,0.098380,0.415657,0.497983
context_recall,0.770833,0.604167,-0.166667,0.416485,0.488546


In [ ]:
# 간단한 통계 해설
print("[정량평가 요약]")
for metric in metric_cols:
    b = summary_mean_df.loc[metric, "baseline_mean"]
    i = summary_mean_df.loc[metric, "improved_mean"]
    d = summary_mean_df.loc[metric, "delta"]
    print(f"- {metric}: baseline={b:.4f}, improved={i:.4f}, delta={d:.4f}")

print()
print("[정성평가 요약]")
for winner, count in judge_summary["count"].items():
    print(f"- {winner}: {count}")


[정량평가 요약]
- faithfulness: baseline=0.5451, improved=0.6135, delta=0.0684
- answer_relevancy: baseline=0.2778, improved=0.2450, delta=-0.0328
- context_precision: baseline=0.4294, improved=0.5278, delta=0.0984
- context_recall: baseline=0.7708, improved=0.6042, delta=-0.1667

[정성평가 요약]
- improved: 15
- baseline: 6
- tie: 3


### 참고

이번 평가는 **수동 평가셋 기반 비교**에 집중

- 이유 : 현재 구현한 Agentic RAG 의 개선 효과를 설명 가능하게 비교하기 위함
- 확장 : 이후에는 RAGAS testset generation 을 추가해 더 큰 평가셋으로 확장 가능

참고 문서
- RAGAS: https://docs.ragas.io/
- Testset generation: https://docs.ragas.io/en/stable/getstarted/rag_testset_generation/
- LangChain OpenAI docs: https://python.langchain.com/docs/integrations/chat/openai/
